# Crush Energy Workbench v7

A streamlined Jupyter UI for deterministic crush-energy, case/subcase, delta-V, statistics, and visualization analysis.

- One primary action: **Save case & calculate**
- A and B are uniform coefficients for the full crush width; lists/ranges mean alternative subcases, not zone values
- Single, comma-list, and inclusive-range input modes
- Case results, 95% confidence intervals for case-set means, crush-profile plots, and exportable statistical plots
- CSV, Excel, JSON, HTML, and PNG exports

> Engineering calculation aid only. Validate coefficient applicability, units, crush measurements, and collision assumptions independently.


In [1]:
# Setup (run once if needed)
# %pip install ipywidgets pandas numpy matplotlib scipy openpyxl pillow


In [2]:
import ast, io, json, math, re, html, base64
from datetime import datetime
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, HTML

G_FTPS2=32.174
LENGTH_TO_FT={"ft":1.0,"in":1/12,"mm":1/304.8,"cm":1/30.48,"m":1/0.3048}
MASS_TO_SLUG={"slug":1.0,"kg":1/14.59390294,"lbm":1/G_FTPS2}
SPEED_FROM_FPS={"ft/s":1.0,"mph":1/1.4666666667,"m/s":1/3.280839895,"km/h":1/0.911344415}
display(HTML("""<style>.cew-title{background:linear-gradient(115deg,#0b253a,#146c86);color:white;padding:18px 22px;border-radius:14px}.cew-note{background:#eef7fa;border-left:4px solid #16809b;padding:10px;border-radius:6px}.cew-warn{background:#fff7e6;border-left:4px solid #d98200;padding:10px;border-radius:6px}.cew-bad{background:#fff0f0;border-left:4px solid #b42318;padding:10px;border-radius:6px}.cew-small{font-size:12px;color:#53636d}</style>"""))

def parse_values(value,name="input"):
    if isinstance(value,(int,float,np.number)): vals=[float(value)]
    else:
        text=str(value).strip()
        if not text: raise ValueError(f"{name} is blank")
        m=re.fullmatch(r"range\(\s*([+-]?\d+)\s*,\s*([+-]?\d+)(?:\s*,\s*([+-]?\d+)\s*)?\)",text)
        if m:
            a,b,h=int(m.group(1)),int(m.group(2)),int(m.group(3) or 1)
            if h==0: raise ValueError(f"{name}: range step cannot be zero")
            if (b-a)*h<0: raise ValueError(f"{name}: step points away from stop")
            vals=[float(v) for v in range(a,b+(1 if h>0 else -1),h)]
            if vals and vals[-1]!=float(b): vals.append(float(b))
        else:
            try: obj=ast.literal_eval(text)
            except Exception:
                try: obj=ast.literal_eval("["+text+"]")
                except Exception as exc: raise ValueError(f"{name}: use Single, Comma list, or Inclusive range") from exc
            vals=[float(obj)] if isinstance(obj,(int,float)) else [float(v) for v in obj]
    if not vals or len(vals)>10000 or not np.all(np.isfinite(vals)): raise ValueError(f"{name}: invalid or excessive values")
    return vals

def mass_slug(v,u):
    if v<=0: raise ValueError("Mass must be greater than zero")
    return v*MASS_TO_SLUG[u]

def to_deg(v,u): return float(v if u=="deg" else (v%12)*30)

def compute_crush(stations,crush,A,B,length_unit="in",include_g=False,overlap_pct=100):
    x=np.asarray(stations,float); c=np.asarray(crush,float)
    if len(x)!=len(c): raise ValueError("Station and crush counts must match")
    if len(x)<2 or np.any(np.diff(x)<=0): raise ValueError("Use at least two strictly increasing stations")
    if np.any(c<0): raise ValueError("Crush cannot be negative")
    if not 0<overlap_pct<=100: raise ValueError("Overlap must be above 0% and at most 100%")
    A=float(A); B=float(B)
    if not np.isfinite(A) or not np.isfinite(B): raise ValueError("A and B must be finite")
    f=LENGTH_TO_FT[length_unit]; xf=x*f; cf=c*f; dx=np.diff(xf); c0,c1=cf[:-1],cf[1:]
    Aft=A/f; Bft=B/(f*f)
    if include_g and Bft==0: raise ValueError("G cannot be included when B is zero")
    G=Aft*Aft/(2*Bft) if include_g else 0.0
    mean_c=(c0+c1)/2; mean_c2=(c0*c0+c0*c1+c1*c1)/3
    energy=(Aft*mean_c+.5*Bft*mean_c2+G)*dx*(overlap_pct/100)
    if np.any(energy<0): raise ValueError("Calculated energy is negative; check A and B")
    return {"energy_ftlb":float(energy.sum()),"avg_crush_ft":float(np.sum(mean_c*dx)/(xf[-1]-xf[0])),"max_crush_ft":float(cf.max()),"x_ft":xf,"c_ft":cf}

def common_impulse(E,m1,m2,e=0):
    if not 0<=e<1: raise ValueError("Restitution must be in [0,1)")
    mu=m1*m2/(m1+m2); closing=math.sqrt(2*E/(mu*(1-e*e))) if E else 0
    rel=(1+e)*closing
    return closing,rel*m2/(m1+m2),rel*m1/(m1+m2)

class ModeInput:
    def __init__(self,label,value,tip="",fixed_list=False):
        self.label=label; inferred="Comma list" if "," in str(value) else "Single"
        opts=["Comma list"] if fixed_list else ["Single","Comma list","Inclusive range"]
        self.mode=W.Dropdown(options=opts,value=inferred if inferred in opts else opts[0],description=f"{label} mode",style={"description_width":"125px"},layout=W.Layout(width="310px"))
        self.single=W.FloatText(value=float(str(value).split(',')[0]),description=label,style={"description_width":"125px"},layout=W.Layout(width="310px"))
        self.list=W.Text(value=str(value),description=label,style={"description_width":"125px"},layout=W.Layout(width="620px"),tooltip=tip)
        self.start=W.IntText(value=0,description="Start",layout=W.Layout(width="150px")); self.stop=W.IntText(value=10,description="Stop",layout=W.Layout(width="150px")); self.step=W.IntText(value=1,description="Step",layout=W.Layout(width="150px"))
        self.editor=W.HBox(); self.widget=W.VBox([self.mode,self.editor]); self.mode.observe(self._draw,'value'); self._draw()
    def _draw(self,*_):
        if self.mode.value=="Single": self.editor.children=[self.single]
        elif self.mode.value=="Comma list": self.editor.children=[self.list]
        else: self.editor.children=[self.start,self.stop,self.step,W.HTML("<span class='cew-small'>Stop included</span>")]
    @property
    def value(self):
        if self.mode.value=="Single": return str(self.single.value)
        if self.mode.value=="Comma list": return self.list.value
        return f"range({self.start.value},{self.stop.value},{self.step.value})"
    @value.setter
    def value(self,v):
        t=str(v); m=re.fullmatch(r"range\(([-+]?\d+),([-+]?\d+),([-+]?\d+)\)",t.replace(' ',''))
        if m and "Inclusive range" in self.mode.options: self.mode.value="Inclusive range"; self.start.value=int(m.group(1)); self.stop.value=int(m.group(2)); self.step.value=int(m.group(3))
        elif ',' in t: self.mode.value="Comma list"; self.list.value=t
        else: self.mode.value="Single"; self.single.value=float(t)
        self._draw()
    @property
    def disabled(self): return self.mode.disabled
    @disabled.setter
    def disabled(self,v):
        for w in [self.mode,self.single,self.list,self.start,self.stop,self.step]: w.disabled=v

def vehicle_panel(name):
    d={"mass":ModeInput("Mass","4000"),"stations":ModeInput("Stations","0,20,40,60",fixed_list=True),"crush":ModeInput("Crush","0,5,10,8",fixed_list=True),"A":ModeInput("A","100","Uniform A; list/range creates alternatives"),"B":ModeInput("B","10","Uniform B; list/range creates alternatives"),"pdof":ModeInput("PDOF","0"),"overlap":ModeInput("Overlap %","100")}
    box=W.Accordion(children=[W.VBox([d[k].widget for k in ["mass","stations","crush","A","B","pdof","overlap"]])],selected_index=0); box.set_title(0,name); d["box"]=box; return d
v1=vehicle_panel("Vehicle 1"); v2=vehicle_panel("Vehicle 2")
mode=W.Dropdown(description="Analysis",options=[("Single vehicle / barrier","single"),("Two vehicles, both crush known","known2"),("Two vehicles, V2 crush unknown","unknown2")],value="single",style={"description_width":"125px"})
combine=W.Dropdown(description="Multiple values",options=[("Every combination","cartesian"),("Match by row","paired")],value="cartesian",style={"description_width":"125px"})
mass_unit=W.Dropdown(description="Mass unit",options=["lbm","kg","slug"],value="lbm",style={"description_width":"125px"}); length_unit=W.Dropdown(description="Length unit",options=["in","ft","mm","cm","m"],value="in",style={"description_width":"125px"}); speed_unit=W.Dropdown(description="Speed output",options=["mph","ft/s","m/s","km/h"],value="mph",style={"description_width":"125px"}); pdof_unit=W.Dropdown(description="PDOF unit",options=["deg","o'clock"],value="deg",style={"description_width":"125px"})
include_g=W.Checkbox(description="Include G = A²/(2B)",value=False); restitution=W.BoundedFloatText(description="Restitution",value=0,min=0,max=.99,step=.05,style={"description_width":"125px"}); unknown_method=W.Dropdown(description="Unknown V2",options=[("Common impulse using V1 energy","thirdlaw"),("Scale V2 energy from V1","scaled")],value="thirdlaw",style={"description_width":"125px"}); energy_factor=ModeInput("V2 energy factor","1")
project=W.Text(description="Project",value="Untitled crush analysis",style={"description_width":"125px"},layout=W.Layout(width="620px")); case_number=W.Text(description="Case number",style={"description_width":"125px"}); analyst=W.Text(description="Analyst",style={"description_width":"125px"}); notes=W.Textarea(description="Notes",style={"description_width":"125px"},layout=W.Layout(width="790px")); case_name=W.Text(description="Case name",value="Case 1",style={"description_width":"125px"})
save_run=W.Button(description="Save case & calculate",button_style="success",icon="play",layout=W.Layout(width="220px")); delete_case=W.Button(description="Delete selected",button_style="warning"); clear_cases=W.Button(description="Clear cases"); run_all=W.Button(description="Recalculate saved cases",button_style="success"); clear_results=W.Button(description="Clear results")
case_select=W.Dropdown(description="Saved cases",options=[],style={"description_width":"90px"}); case_out=W.Output(); status=W.HTML(); results_out=W.Output(); stats_out=W.Output(); vis_out=W.Output(); export_out=W.Output()
state={"results":pd.DataFrame(),"profiles":[],"rejected":pd.DataFrame(),"config":{}}; cases=[]

def update_mode(*_):
    v2["box"].layout.display="none" if mode.value=="single" else ""
    for k in ["stations","crush","A","B"]: v2[k].disabled=mode.value=="unknown2"
    unknown_method.layout.display="" if mode.value=="unknown2" else "none"; energy_factor.widget.layout.display="" if mode.value=="unknown2" and unknown_method.value=="scaled" else "none"
mode.observe(update_mode,'value'); unknown_method.observe(update_mode,'value'); update_mode()

def snapshot():
    d={"case_name":case_name.value.strip() or f"Case {len(cases)+1}","mode":mode.value,"combine":combine.value,"mass_unit":mass_unit.value,"length_unit":length_unit.value,"speed_unit":speed_unit.value,"pdof_unit":pdof_unit.value,"include_g":include_g.value,"restitution":restitution.value,"unknown_method":unknown_method.value,"energy_factor":energy_factor.value}
    for prefix,v in [("v1",v1),("v2",v2)]:
        for k in ["mass","stations","crush","A","B","pdof","overlap"]: d[f"{prefix}_{k}"]=v[k].value
    return d

def render_cases():
    case_select.options=[(d["case_name"],i) for i,d in enumerate(cases)]; case_out.clear_output()
    with case_out:
        if cases: display(pd.DataFrame(cases)[["case_name","mode","v1_mass","v1_A","v1_B","v2_mass"]])

def combinations(p,method):
    names=list(p); vals=[p[k] for k in names]
    if method=="cartesian": return [dict(zip(names,row)) for row in product(*vals)]
    n=max(map(len,vals)); bad=[k for k,v in p.items() if len(v) not in (1,n)]
    if bad: raise ValueError("Match by row requires each multiple-value input to have the same length: "+", ".join(bad))
    return [{k:(v[0] if len(v)==1 else v[i]) for k,v in p.items()} for i in range(n)]

def parameters(d):
    p={"m1":parse_values(d["v1_mass"],"V1 mass"),"A1":parse_values(d["v1_A"],"V1 A"),"B1":parse_values(d["v1_B"],"V1 B"),"p1":parse_values(d["v1_pdof"],"V1 PDOF"),"o1":parse_values(d["v1_overlap"],"V1 overlap")}
    if d["mode"]!="single": p.update({"m2":parse_values(d["v2_mass"],"V2 mass"),"p2":parse_values(d["v2_pdof"],"V2 PDOF"),"o2":parse_values(d["v2_overlap"],"V2 overlap")})
    if d["mode"]=="known2": p.update({"A2":parse_values(d["v2_A"],"V2 A"),"B2":parse_values(d["v2_B"],"V2 B")})
    if d["mode"]=="unknown2" and d["unknown_method"]=="scaled": p["factor"]=parse_values(d["energy_factor"],"V2 factor")
    return combinations(p,d["combine"])

def calculate(*_):
    results_out.clear_output(); stats_out.clear_output(); vis_out.clear_output(); export_out.clear_output(); rows=[]; profiles=[]; rejected=[]; cid=0
    for d in cases:
        try: subs=parameters(d)
        except Exception as e: rejected.append({"case_name":d["case_name"],"reason":str(e)}); continue
        for si,c in enumerate(subs,1):
            cid+=1
            try:
                m1=mass_slug(c["m1"],d["mass_unit"]); cp1=compute_crush(parse_values(d["v1_stations"],"V1 stations"),parse_values(d["v1_crush"],"V1 crush"),c["A1"],c["B1"],d["length_unit"],d["include_g"],c["o1"]); E1=cp1["energy_ftlb"]; bs1=math.sqrt(2*E1/m1); sf=SPEED_FROM_FPS[d["speed_unit"]]
                E2=0; cp2=None; coll=None; est=False
                if d["mode"]=="known2":
                    m2=mass_slug(c["m2"],d["mass_unit"]); cp2=compute_crush(parse_values(d["v2_stations"],"V2 stations"),parse_values(d["v2_crush"],"V2 crush"),c["A2"],c["B2"],d["length_unit"],d["include_g"],c["o2"]); E2=cp2["energy_ftlb"]; coll=common_impulse(E1+E2,m1,m2,d["restitution"])
                elif d["mode"]=="unknown2":
                    m2=mass_slug(c["m2"],d["mass_unit"]); est=True
                    if d["unknown_method"]=="scaled": E2=E1*c["factor"]*c["o2"]/c["o1"]
                    coll=common_impulse(E1+E2,m1,m2,d["restitution"])
                dv1=coll[1] if coll else bs1; a1=math.radians(to_deg(c["p1"],d["pdof_unit"])); row={"case_id":cid,"case_name":d["case_name"],"subcase":si,"estimated_unknown":est,f"v1_mass_{d['mass_unit']}":c["m1"],"v1_A":c["A1"],"v1_B":c["B1"],"v1_energy_ftlb":E1,f"v1_barrier_speed_{d['speed_unit']}":bs1*sf,f"v1_delta_v_{d['speed_unit']}":dv1*sf,f"v1_longitudinal_{d['speed_unit']}":dv1*sf*math.cos(a1),f"v1_lateral_{d['speed_unit']}":dv1*sf*math.sin(a1)}
                profiles.append({"case_id":cid,"case_name":d["case_name"],"vehicle":"V1","x":cp1["x_ft"],"c":cp1["c_ft"]})
                if coll:
                    a2=math.radians(to_deg(c["p2"],d["pdof_unit"])); row.update({f"v2_mass_{d['mass_unit']}":c["m2"],"v2_energy_ftlb":E2,"system_energy_ftlb":E1+E2,f"closing_speed_{d['speed_unit']}":coll[0]*sf,f"v2_delta_v_{d['speed_unit']}":coll[2]*sf,f"v2_longitudinal_{d['speed_unit']}":coll[2]*sf*math.cos(a2),f"v2_lateral_{d['speed_unit']}":coll[2]*sf*math.sin(a2)})
                    if cp2: row.update({"v2_A":c["A2"],"v2_B":c["B2"],f"v2_barrier_speed_{d['speed_unit']}":math.sqrt(2*E2/m2)*sf}); profiles.append({"case_id":cid,"case_name":d["case_name"],"vehicle":"V2","x":cp2["x_ft"],"c":cp2["c_ft"]})
                rows.append(row)
            except Exception as e: rejected.append({"case_id":cid,"case_name":d["case_name"],"subcase":si,"reason":str(e)})
    state.update({"results":pd.DataFrame(rows),"profiles":profiles,"rejected":pd.DataFrame(rejected),"config":{"project":project.value,"case_number":case_number.value,"analyst":analyst.value,"notes":notes.value,"created":datetime.now().isoformat(timespec='seconds')}})
    if rows: status.value=f"<div class='cew-note'>{len(rows)} result row(s) updated.</div>"; render_results(); render_visuals()
    else: status.value="<div class='cew-bad'>No valid results. Review rejected cases.</div>"; render_results()

def save_calculate(_=None):
    d=snapshot(); hit=next((i for i,x in enumerate(cases) if x["case_name"]==d["case_name"]),None)
    if hit is None: cases.append(d); case_name.value=f"Case {len(cases)+1}"
    else: cases[hit]=d
    render_cases(); calculate()
save_run.on_click(save_calculate); run_all.on_click(calculate)
def delete_selected(_):
    if cases and case_select.value is not None: cases.pop(int(case_select.value)); render_cases(); calculate()
def clear_all_cases(_): cases.clear(); render_cases(); state["results"]=pd.DataFrame(); render_results()
delete_case.on_click(delete_selected); clear_cases.on_click(clear_all_cases)

def statistics_table():
    rows=[]
    if state["results"].empty: return pd.DataFrame()
    for col in state["results"].select_dtypes(include=np.number).columns:
        if col in ("case_id","subcase"): continue
        x=state["results"][col].dropna().astype(float); n=len(x); mean=x.mean(); sd=x.std(ddof=1) if n>1 else np.nan
        if n>1:
            from scipy.stats import t; margin=t.ppf(.975,n-1)*sd/math.sqrt(n); lo,hi=mean-margin,mean+margin
        else: lo=hi=np.nan
        rows.append({"metric":col,"n":n,"mean":mean,"std_dev":sd,"std_error":sd/math.sqrt(n) if n>1 else np.nan,"95% CI mean lower":lo,"95% CI mean upper":hi,"min":x.min(),"median":x.median(),"max":x.max()})
    return pd.DataFrame(rows)

def stats_png():
    st=statistics_table(); df=state["results"]; targets=[c for c in df.select_dtypes(include=np.number) if any(k in c for k in ["energy","delta_v","closing_speed","barrier_speed"])][:8]
    fig,ax=plt.subplots(2,1,figsize=(11,8)); q=st.set_index("metric").reindex(targets); means=q["mean"].to_numpy(); lo=q["95% CI mean lower"].to_numpy(); hi=q["95% CI mean upper"].to_numpy(); err=np.vstack([np.nan_to_num(means-lo),np.nan_to_num(hi-means)]); ax[0].errorbar(range(len(targets)),means,yerr=err,fmt='o',capsize=5); ax[0].set_xticks(range(len(targets)),targets,rotation=35,ha='right'); ax[0].set_title('Means with 95% confidence intervals'); ax[0].grid(True,alpha=.25); ax[1].boxplot([df[c].dropna() for c in targets],label=targets,showmeans=True); ax[1].tick_params(axis='x',rotation=35); ax[1].set_title('Case-set distributions'); fig.tight_layout(); b=io.BytesIO(); fig.savefig(b,format='png',dpi=180); plt.close(fig); return b.getvalue()

def render_results():
    results_out.clear_output(); stats_out.clear_output()
    with results_out:
        if not state["results"].empty: display(state["results"].round(4))
        if not state["rejected"].empty: display(HTML('<h4>Rejected cases</h4>')); display(state["rejected"])
    with stats_out:
        if not state["results"].empty:
            display(HTML("<div class='cew-warn'>95% CI is for the arithmetic mean across evaluated cases/subcases, not a reconstruction uncertainty interval.</div>")); display(statistics_table().round(4)); display(HTML(f"<img style='max-width:100%' src='data:image/png;base64,{base64.b64encode(stats_png()).decode()}'>"))

def render_visuals(*_):
    vis_out.clear_output()
    with vis_out:
        if not state["profiles"]: display(HTML("<div class='cew-warn'>Save and calculate a case first.</div>")); return
        fig,ax=plt.subplots(1,2,figsize=(12,4));
        for p in state["profiles"]: ax[0].plot(p["x"],p["c"],marker='o',label=f"{p['case_name']} {p['vehicle']} ({p['case_id']})")
        ax[0].set(title='Residual crush profiles',xlabel='Station (ft)',ylabel='Crush (ft)'); ax[0].grid(True,alpha=.25)
        if len(state["profiles"])<=12: ax[0].legend(fontsize=8)
        df=state["results"]; ax[1].bar(df["case_id"].astype(str),df["v1_energy_ftlb"]); ax[1].set(title='Vehicle 1 energy by result',xlabel='Result',ylabel='ft·lbf'); fig.tight_layout(); plt.show()

def require():
    if state["results"].empty: raise ValueError("Calculate a case first")
def csv_bytes(): require(); return state["results"].to_csv(index=False).encode()
def json_bytes(): require(); return json.dumps({"config":state["config"],"results":state["results"].to_dict('records'),"rejected":state["rejected"].to_dict('records')},indent=2,default=str).encode()
def xlsx_bytes():
    require(); bio=io.BytesIO(); from openpyxl.drawing.image import Image as XLImage
    with pd.ExcelWriter(bio,engine='openpyxl') as w:
        pd.DataFrame([state["config"]]).to_excel(w,'Project',index=False); state["results"].to_excel(w,'Case Results',index=False); statistics_table().to_excel(w,'Statistics',index=False); state["rejected"].to_excel(w,'Rejected Cases',index=False); ws=w.book.create_sheet('Statistical Plots'); ws.add_image(XLImage(io.BytesIO(stats_png())),'A1')
    return bio.getvalue()
def html_bytes():
    require(); im=base64.b64encode(stats_png()).decode(); return f"<h1>{html.escape(project.value)}</h1><p>95% CI is for the case-set mean, not reconstruction uncertainty.</p>{state['results'].to_html(index=False)}<h2>Statistics</h2>{statistics_table().to_html(index=False)}<img style='max-width:100%' src='data:image/png;base64,{im}'>".encode()
exp_csv=W.Button(description='CSV'); exp_xlsx=W.Button(description='Excel + plots'); exp_json=W.Button(description='JSON'); exp_html=W.Button(description='HTML + plots'); exp_png=W.Button(description='Stats PNG')
def exporter(fn,name,mime):
    def go(_):
        export_out.clear_output()
        try:
            data=fn(); link=f"<a download='{name}' href='data:{mime};base64,{base64.b64encode(data).decode()}'>Download {name}</a>"
            with export_out: display(HTML(link))
        except Exception as e:
            with export_out: display(HTML(f"<div class='cew-bad'>{html.escape(str(e))}</div>"))
    return go
for b,fn,n,m in [(exp_csv,csv_bytes,'case_results.csv','text/csv'),(exp_xlsx,xlsx_bytes,'crush_workbench.xlsx','application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'),(exp_json,json_bytes,'crush_project.json','application/json'),(exp_html,html_bytes,'crush_report.html','text/html'),(exp_png,stats_png,'statistics.png','image/png')]: b.on_click(exporter(fn,n,m))

help_html=W.HTML(r"""<div class='cew-note'><h2>Help</h2><p><b>Workflow:</b> enter inputs, name the case, and click Save case & calculate. Reusing a name updates that case and refreshes all outputs.</p><p><b>A and B:</b> each is uniform across the entire measured crush width. Single uses one coefficient. Comma list or Inclusive range means alternative coefficient assumptions and creates subcases. Values are never interpreted as zone-by-zone coefficients.</p><p><b>Stations and crush:</b> use comma lists with equal counts and strictly increasing stations.</p><p><b>Inclusive range:</b> stop is always included; if the step misses it, stop is appended.</p><p><b>Energy:</b> F=A+BC and e=AC+½BC². The program analytically integrates C and C² between adjacent stations assuming linear crush variation.</p><p><b>95% CI:</b> Student-t interval for the arithmetic mean across evaluated cases/subcases. It is not a prediction interval, tolerance interval, or reconstruction uncertainty interval.</p><p><b>Limits:</b> verify stiffness applicability and units. Residual crush may omit restitution and dynamic deformation. Two-vehicle outputs use a collinear common-impulse model and do not solve rotation, trajectory, tire forces, or post-impact motion.</p></div>""")
header=W.HTML("<div class='cew-title'><h2>Crush Energy Workbench</h2><p>Cases, subcases, energy, delta-V, statistics, and visualizations</p></div>")
settings=W.VBox([W.HBox([mode,combine]),W.HBox([mass_unit,length_unit]),W.HBox([speed_unit,pdof_unit]),W.HBox([include_g,restitution]),unknown_method,energy_factor.widget])
inputs=W.VBox([W.HTML("<div class='cew-note'>A and B are uniform coefficients. Lists and inclusive ranges create alternative subcases.</div>"),project,W.HBox([case_number,analyst]),notes,settings,v1["box"],v2["box"],case_name,save_run,status])
cases_page=W.VBox([case_select,W.HBox([delete_case,clear_cases,run_all]),case_out])
results_page=W.VBox([results_out]); stats_page=W.VBox([stats_out]); visuals_page=W.VBox([vis_out]); exports=W.VBox([W.HBox([exp_csv,exp_xlsx,exp_json,exp_html,exp_png]),export_out])
tabs=W.Tab(children=[inputs,cases_page,results_page,stats_page,visuals_page,exports,help_html])
for i,t in enumerate(["Setup & Inputs","Cases","Results","Statistics","Visualizations","Exports","Help"]): tabs.set_title(i,t)
display(W.VBox([header,tabs]))
